# 18 - Claims Lakehouse: Pseudocode by Stage

Short pseudocode for each stage of the Workflows DAG, from Auto Loader to the gatekeeper.
Run order: Bronze → Validate + dedup → Silver MERGE → Gold → Ledger → Gatekeeper.
Upstream (not shown): 837 EDI → JSON/CSV in ADLS Gen2, ADF triggers the Workflow.

## 1. Bronze ingest (Auto Loader)
```python
stream = read cloudFiles(format=json, schema=claims_raw_schema,
                         maxFilesPerTrigger=50)          # directory listing mode
            .from(landing_path)
add columns: _ingest_timestamp = now(), _source_file = input_file_name()
write stream to bronze.claims_raw (Delta)
      checkpoint = checkpoint_path                      # one per table
      trigger    = availableNow                         # process new files, then stop
```

## 2. Validate + deduplicate (one pass)
```python
batch = new rows from bronze since last watermark

rejection_code = CASE
    claim_id IS NULL          -> 'ERR_NULL_PK'
    amount < 0                -> 'ERR_NEGATIVE_AMOUNT'
    service_date > today      -> 'ERR_FUTURE_SERVICE_DATE'

quarantined = rows WHERE rejection_code IS NOT NULL  -> write quarantine table
valid       = rows WHERE rejection_code IS NULL       # split BEFORE ranking

rn = row_number() OVER (PARTITION BY claim_id
        ORDER BY updated_at DESC, _source_file DESC, _file_modified_at DESC)  # deterministic

curated    = valid WHERE rn = 1     -> goes to MERGE
duplicates = valid WHERE rn > 1     -> keep rows + amounts for the ledger
```

## 3. Silver MERGE
```sql
MERGE INTO silver.claims t
USING curated s
ON  t.claim_id = s.claim_id AND t.service_year_month = s.service_year_month
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *   -- rerun-safe
WHEN NOT MATCHED THEN INSERT *;

-- table: PARTITIONED BY service_year_month
OPTIMIZE silver.claims ZORDER BY (provider_id);   -- compact small files
advance watermark
```

## 4. Gold models (grain-safe)
```python
lines_per_claim = claim_services GROUP BY claim_id
                  -> line_count, SUM(line_billed_amount)   # roll lines up FIRST

gold = claims
       LEFT JOIN lines_per_claim ON claim_id                # 1:1 now, no fan-out
       JOIN broadcast(providers) ON provider_id             # small dim
       GROUP BY provider_id, service_year_month

AQE on: coalesce partitions, skew join (big hospital networks)

grain tests (must return 0 rows):
    claims         GROUP BY claim_id              HAVING COUNT(*) > 1
    claim_services GROUP BY claim_id, line_number HAVING COUNT(*) > 1
    SUM(gold paid) != SUM(header paid)
```

## 5. Reconciliation ledger
```python
ledger = union(curated     as 'curated',
               quarantined as 'quarantined',
               duplicates  as 'duplicate')
         GROUP BY bucket -> COUNT(*), SUM(amount)       # one pass, DECIMAL amounts

unexplained = source_gross - (curated + quarantined + duplicates)
write ledger row (batch_id, buckets, unexplained) -> audit table
# must be 0.00 for finance to sign off the load
```

## 6. Gatekeeper (runs last)
```python
metrics = [row_count, key_null_rate, duplicate_ratio, freshness_lag,
           schema_drift, quarantine_rate, financial_variance,
           run_duration_drift, shuffle_spill]

if every metric within threshold:
    trigger BI refresh                 # Gold certified by 6:00 AM
else:
    hold BI refresh                    # Gold already written, just not published
    page on-call
```

## On failure
```text
Delta commits are atomic -> no partial writes
Repair-and-Rerun          -> reruns only the failed task + dependents
Even if MERGE reruns      -> dedup + updated_at guard = same result
```